# Peace Capital — Spot Pricer · Quantile Layer
### P10 / P50 / P90 Prediction Intervals · Regime-Conditional
---
**Extends:** `pc_spot_pricer.ipynb`  
**Purpose:** Replace single point forecast with a full uncertainty envelope.  
**Output:** For each delivery hour → low (P10), central (P50), high (P90) price forecast + confidence width as a position-sizing signal.

**Ref:** PC-PWR-2026-PROG02

---
### What This Adds

```
Point Forecast (PROG01)
    └──▶  single price estimate per hour

Quantile Layer (PROG02)
    ├── P10  lower bound — price is above this 90% of the time
    ├── P50  median forecast (replaces point estimate)
    ├── P90  upper bound — price is below this 90% of the time
    ├── Width = P90 − P10  →  uncertainty signal
    └── Position multiplier = f(regime conviction, interval width)
```

> **Run PROG01 first.** This notebook loads its artifacts and extends them. It accepts `pc_spot_pricer.pkl` or `pc_spot_pricer_real.pkl`. You do not need to re-run the full simulation.

---
## 0. Environment & Load Artifacts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import pickle
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import mean_absolute_error

from peace_power_market_toolkit import (
    add_bess_features,
    estimate_storage_sunset,
    simulate_bess_fleet,
)

np.random.seed(42)

PC = dict(
    navy='#0e1829', navy2='#121f33', navy3='#192740',
    teal='#3d9e9e', teal_d='#1a5c5c',
    crimson='#c0283e', gold='#c9a84c',
    text='#dde4ed', text2='#a8b8cc', text3='#677a90',
    rule='#1e3050', rule2='#243855',
    green='#22c55e', amber='#f59e0b',
)

plt.rcParams.update({
    'figure.facecolor': PC['navy'],
    'axes.facecolor':   PC['navy2'],
    'axes.edgecolor':   PC['rule'],
    'axes.labelcolor':  PC['text3'],
    'xtick.color':      PC['text3'],
    'ytick.color':      PC['text3'],
    'text.color':       PC['text'],
    'grid.color':       PC['rule'],
    'grid.alpha':       0.4,
    'font.family':      'serif',
    'axes.titlecolor':  PC['text'],
    'axes.titlesize':   11,
    'legend.facecolor': PC['navy2'],
    'legend.edgecolor': PC['rule'],
    'legend.labelcolor':PC['text2'],
})

# ── Load PROG01 artifacts ─────────────────────────────────────
artifact_path = None
for candidate in ['pc_spot_pricer.pkl', 'pc_spot_pricer_real.pkl']:
    try:
        with open(candidate, 'rb') as f:
            arts = pickle.load(f)
        artifact_path = candidate
        break
    except FileNotFoundError:
        continue

if artifact_path is None:
    raise FileNotFoundError(
        'Missing PROG01 artifacts. Expected pc_spot_pricer.pkl or pc_spot_pricer_real.pkl.'
    )

required_keys = {'regime_clf', 'scaler_reg', 'regime_models', 'blend_model',
                 'regime_features', 'price_features'}
if not required_keys.issubset(set(arts.keys())):
    missing = sorted(required_keys.difference(set(arts.keys())))
    raise KeyError(
        'Loaded artifact is not PROG01-compatible. Missing keys: '
        + ', '.join(missing)
    )

regime_clf      = arts['regime_clf']
scaler_reg      = arts['scaler_reg']
regime_models   = arts['regime_models']   # point forecast models
blend_model     = arts['blend_model']
REGIME_FEATURES = arts['regime_features']
PRICE_FEATURES  = arts['price_features']

regime_names = ['Thermal-Marginal', 'Renewable-Dominant', 'Demand-Stress']

# Reload test predictions from PROG01 if available (informational only)
try:
    test_pred = pd.read_csv('pc_test_predictions.csv', index_col=0, parse_dates=True)
    test_rows_msg = f'{len(test_pred):,}'
except FileNotFoundError:
    test_pred = pd.DataFrame()
    test_rows_msg = 'n/a (pc_test_predictions.csv not found)'

print('✓ PROG01 artifacts loaded')
print(f'  Artifact path : {artifact_path}')
print(f'  Test set rows : {test_rows_msg}')
print(f'  Price features: {len(PRICE_FEATURES)}')
print(f'  Regime models : {list(regime_models.keys())}')



---
## I. Rebuild Data for Quantile Training

We need the full feature matrix from PROG01. Re-run the simulation and feature engineering quickly — same seed, identical data.

In [ ]:
# ── Re-simulate with storage-aware feature engineering ────────
np.random.seed(42)
START  = '2023-01-01'
N_DAYS = 730
MAX_ATC = 3000

STORAGE_FEATURES = [
    'bess_online_mw', 'bess_absorption_mw', 'bess_discharge_mw',
    'renewable_surplus_mw', 'surplus_after_bess_mw',
    'peak_gap_mw', 'peak_gap_after_bess_mw',
    'bess_capture_ratio', 'bess_peak_relief_ratio',
    'storage_penetration', 'rsi_after_bess', 'rsi_delta_bess',
    'rsi_storage_decay', 'short_edge_score', 'storage_sunset_flag',
    'ttf_x_rsi_after_bess', 'storage_x_surplus', 'storage_x_peak_gap',
    'price_vs_bess_net_surplus',
]


def simulate_spanish_market(n_days, start):
    n   = n_days * 24
    idx = pd.date_range(start, periods=n, freq='h')
    hour      = idx.hour
    dayofweek = idx.dayofweek
    month     = idx.month
    is_weekend= dayofweek >= 5
    is_peak   = (hour >= 8) & (hour < 20) & ~is_weekend

    ttf_d  = np.clip(35 + np.cumsum(np.random.normal(0, 0.4, n_days)), 15, 120)
    eua_d  = np.clip(65 + np.cumsum(np.random.normal(0.02, 0.5, n_days)), 40, 110)
    brt_d  = np.clip(80 + 0.6*(ttf_d-35) + np.random.normal(0,2,n_days), 50, 130)
    ttf, eua, brent = np.repeat(ttf_d,24), np.repeat(eua_d,24), np.repeat(brt_d,24)
    thermal_floor = ttf * 0.45 + eua * 0.35

    temp_s  = 15 + 12*np.sin(2*np.pi*(idx.dayofyear/365 - 0.25))
    temperature = temp_s + np.random.normal(0, 2, n)
    solar_noon  = 13
    daylight    = 6 + 4*np.sin(2*np.pi*(idx.dayofyear/365 - 0.25))
    ghi = np.clip(
        800*np.exp(-0.5*((hour-solar_noon)/(daylight/2))**2)
        *(1+0.15*np.sin(2*np.pi*idx.dayofyear/365))
        + np.random.normal(0,30,n), 0, 1100)
    wind_speed = np.clip(
        np.abs(6+2*np.cos(2*np.pi*idx.dayofyear/365)+np.random.normal(0,2.5,n)), 0, 25)

    solar_gen = np.clip(ghi/1000*22 + np.random.normal(0,.5,n), 0, 22)
    wind_gen  = np.clip(wind_speed**1.5/15*30 + np.random.normal(0,1,n), 0, 30)
    nuclear   = np.full(n, 7.2) + np.random.normal(0,.2,n)
    hydro     = np.clip(5+3*np.sin(2*np.pi*(idx.dayofyear/365+0.3))+np.random.normal(0,.8,n),.5,12)

    d_base = 28 - 8*np.sin(2*np.pi*(idx.dayofyear/365-.1))
    d_hour = 1 + .3*np.sin(np.pi*(hour-6)/14)*is_peak.astype(float)
    d_temp = .4*np.abs(temperature-18)
    demand = np.clip(d_base*d_hour + d_temp + np.random.normal(0,.6,n), 18, 48)
    ren_gen= solar_gen + wind_gen
    rsi    = ren_gen / demand
    gas_gen= np.clip(demand - solar_gen - wind_gen - nuclear - hydro, 0, 25)

    base = pd.DataFrame({
        'price_omie': np.nan,
        'ttf': ttf,
        'eua': eua,
        'brent': brent,
        'thermal_floor': thermal_floor,
        'temperature': temperature,
        'ghi': ghi,
        'wind_speed': wind_speed,
        'gen_solar': solar_gen,
        'gen_wind': wind_gen,
        'gen_nuclear': nuclear,
        'gen_hydro': hydro,
        'gen_gas': gas_gen,
        'demand': demand,
        'rsi': rsi,
        'hour': hour,
        'dayofweek': dayofweek,
        'month': month,
        'is_weekend': is_weekend.astype(int),
        'is_peak': is_peak.astype(int),
    }, index=idx)
    base = simulate_bess_fleet(base, demand_col='demand')

    rsi_after_bess = (
        (ren_gen - base['bess_absorption_mw'].to_numpy())
        / np.clip(demand, 1, None)
    )
    storage_scale = max(
        float(np.nanquantile(
            np.maximum(np.clip(ren_gen - demand, 0, None), base['bess_online_mw'].to_numpy()),
            0.95,
        )),
        1.0,
    )
    storage_penetration = base['bess_online_mw'].to_numpy() / storage_scale
    storage_drag = np.clip(1.0 - storage_penetration / 0.30, 0.15, 1.0)

    true_regime = np.zeros(n, dtype=int)
    true_regime[rsi_after_bess > 0.65] = 1
    true_regime[((temperature < 5)|(temperature > 35)) & is_peak] = 2

    price = np.zeros(n)
    m0 = true_regime == 0
    price[m0] = thermal_floor[m0] + 5*is_peak[m0] + np.random.normal(0,4,m0.sum())
    m1 = true_regime == 1
    price[m1] = (
        thermal_floor[m1]
        - np.clip(rsi_after_bess[m1] - 0.65, 0, None) * 60 * storage_drag[m1]
        + np.random.normal(0,5,m1.sum())
    )
    m2 = true_regime == 2
    price[m2] = (
        thermal_floor[m2]
        + 40 + 20*np.abs(temperature[m2]-18)/18
        - 0.25 * base.loc[m2, 'bess_discharge_mw'].to_numpy()
        + np.random.normal(0,8,m2.sum())
    )
    price = np.clip(price, -100, 300)

    base['price_omie'] = price
    base['rsi_after_bess'] = rsi_after_bess
    base['storage_penetration_proxy'] = storage_penetration
    base['true_regime'] = true_regime
    return base


def build_features(df):
    f = df.copy()
    window = 20*24
    for fuel in ['ttf','eua','brent']:
        rc = f['price_omie'].rolling(window).cov(f[fuel])
        rv = f[fuel].rolling(window).var()
        f[f'beta_{fuel}'] = (rc/rv).clip(-5,5)
    f['price_vs_floor']  = f['price_omie'] - f['thermal_floor']
    f['floor_ratio']     = f['price_omie'] / f['thermal_floor'].clip(1)
    f['ren_total']       = f['gen_solar'] + f['gen_wind']
    f['ren_share']       = f['ren_total'] / f['demand'].clip(1)
    f['solar_share']     = f['gen_solar'] / f['demand'].clip(1)
    f['wind_share']      = f['gen_wind']  / f['demand'].clip(1)
    f['thermal_slack']   = f['demand']-f['gen_nuclear']-f['gen_hydro']-f['ren_total']
    f['temp_deviation']  = np.abs(f['temperature']-18)
    f['temp_sq']         = f['temperature']**2
    f['demand_rolling']  = f['demand'].rolling(48).mean()
    f['demand_z']        = ((f['demand']-f['demand_rolling'])
                            /f['demand'].rolling(48).std().clip(.1))
    for lag in [1,2,3,24,48,168]:
        f[f'price_lag_{lag}h'] = f['price_omie'].shift(lag)
    f['price_roll24_mean'] = f['price_omie'].shift(24).rolling(24).mean()
    f['price_roll24_std']  = f['price_omie'].shift(24).rolling(24).std()
    f['price_roll168_mean']= f['price_omie'].shift(24).rolling(168).mean()
    f['hour_sin']  = np.sin(2*np.pi*f['hour']/24)
    f['hour_cos']  = np.cos(2*np.pi*f['hour']/24)
    f['month_sin'] = np.sin(2*np.pi*f['month']/12)
    f['month_cos'] = np.cos(2*np.pi*f['month']/12)
    f['dow_sin']   = np.sin(2*np.pi*f['dayofweek']/7)
    f['dow_cos']   = np.cos(2*np.pi*f['dayofweek']/7)
    f['ttf_x_rsi']   = f['ttf']*f['rsi']
    f['temp_x_peak'] = f['temp_deviation']*f['is_peak']
    f['wind_x_solar']= f['gen_wind']*f['gen_solar']
    f = add_bess_features(f)
    return f


df      = simulate_spanish_market(N_DAYS, START)
df_feat = build_features(df).dropna()
PRICE_FEATURES = list(dict.fromkeys(PRICE_FEATURES + [
    feature for feature in STORAGE_FEATURES if feature in df_feat.columns
]))

# Add regime probabilities from loaded classifier
all_X_s = scaler_reg.transform(df_feat[REGIME_FEATURES])
rp = regime_clf.predict_proba(all_X_s)
df_feat['pred_regime']    = regime_clf.predict(all_X_s)
df_feat['prob_thermal']   = rp[:, 0]
df_feat['prob_renewable'] = rp[:, 1]
df_feat['prob_stress']    = rp[:, 2]

SPLIT_DATE = '2024-01-01'
train_f = df_feat[df_feat.index < SPLIT_DATE]
test_f  = df_feat[df_feat.index >= SPLIT_DATE]

print(f'✓ Data rebuilt — {len(df_feat):,} hours')
print(f'  Train: {len(train_f):,}  |  Test: {len(test_f):,}')
print(f'  Storage-aware price features: {len([f for f in STORAGE_FEATURES if f in PRICE_FEATURES])}')



---
## II. Quantile XGBoost — Theory

Standard XGBoost minimises MSE — it finds the conditional **mean**. Quantile regression changes the loss function to find any percentile of the conditional distribution.

For quantile `q`, the loss function is:

```
L(y, ŷ) = q × max(y − ŷ, 0)  +  (1−q) × max(ŷ − y, 0)
```

- **q = 0.10** → model learns to underestimate 90% of the time → lower bound
- **q = 0.50** → model learns the median → replaces point estimate  
- **q = 0.90** → model learns to overestimate 90% of the time → upper bound

The **width** `P90 − P10` is your uncertainty signal. Narrow = high conviction. Wide = reduce position or pass.

We train **three models per regime** (9 models total), so each regime has its own calibrated uncertainty envelope.

---
## III. Train Quantile Models

In [ ]:
QUANTILES = [0.10, 0.50, 0.90]
Q_LABELS  = ['p10', 'p50', 'p90']
TARGET    = 'price_omie'

BASE_PARAMS = dict(
    n_estimators    = 500,
    learning_rate   = 0.03,
    max_depth       = 6,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    min_child_weight= 5,
    reg_alpha       = 0.1,
    reg_lambda      = 1.0,
    random_state    = 42,
    n_jobs          = -1,
    # Quantile-specific
    objective       = 'reg:quantileerror',
)

# quantile_models[regime][q_label] = fitted XGBRegressor
quantile_models = {r: {} for r in [0, 1, 2]}

for regime in [0, 1, 2]:
    tr = train_f[train_f['true_regime'] == regime]
    te = test_f[test_f['true_regime']   == regime]

    print(f'\nRegime {regime} — {regime_names[regime]}')
    print(f'  Train: {len(tr):,}  |  Test: {len(te):,}')

    for q, label in zip(QUANTILES, Q_LABELS):
        m = xgb.XGBRegressor(**BASE_PARAMS, quantile_alpha=q)
        m.fit(tr[PRICE_FEATURES], tr[TARGET], verbose=False)
        quantile_models[regime][label] = m

        # Calibration check: coverage should match quantile
        preds    = m.predict(te[PRICE_FEATURES])
        coverage = (te[TARGET].values <= preds).mean()
        print(f'  {label} (q={q}) → coverage {coverage:.1%}  '
              f'(target {q:.0%})')

print('\n✓ All quantile models trained')

---
## IV. Calibration Check

A well-calibrated P10/P90 interval should contain the actual price 80% of the time. We check this per regime and overall — miscalibration means the intervals are overconfident or too wide.

In [ ]:
def score_quantile_forecast(test_df, models, features, target):
    """
    Returns a DataFrame with p10, p50, p90 predictions
    and coverage / width diagnostics.
    """
    out = test_df[[target, 'true_regime', 'pred_regime']].copy()

    for label in Q_LABELS:
        preds = np.full(len(test_df), np.nan)
        for regime, regime_m in models.items():
            mask = test_df['pred_regime'].values == regime
            if mask.sum() > 0 and label in regime_m:
                preds[mask] = regime_m[label].predict(
                    test_df.loc[mask, features])
        out[label] = preds

    out['width']     = out['p90'] - out['p10']
    out['in_band']   = ((out[target] >= out['p10'])
                        & (out[target] <= out['p90'])).astype(int)
    out['p50_error'] = out[target] - out['p50']
    return out.dropna()


results = score_quantile_forecast(test_f, quantile_models,
                                   PRICE_FEATURES, TARGET)

print('─' * 55)
print('  Calibration Report — P10/P90 Interval')
print('─' * 55)
overall_cov = results['in_band'].mean()
print(f'  Overall coverage  : {overall_cov:.1%}  (target 80%)')
print(f'  Mean interval width: {results["width"].mean():.1f} €/MWh')
print(f'  Median P50 MAE    : {results["p50_error"].abs().mean():.2f} €/MWh')
print()
print('  Coverage by regime:')
for r, name in enumerate(regime_names):
    sub = results[results['true_regime'] == r]
    if len(sub) == 0: continue
    cov = sub['in_band'].mean()
    w   = sub['width'].mean()
    mae = sub['p50_error'].abs().mean()
    print(f'  {name:<22}: coverage {cov:.1%}  '
          f'width {w:.1f} €/MWh  MAE {mae:.1f}')
print('─' * 55)

In [ ]:
# ── Coverage calibration plot ────────────────────────────────
# Shows what fraction of actuals fall below each quantile forecast
# Perfect calibration = diagonal line

quantile_levels = np.arange(0.05, 1.0, 0.05)
empirical_coverage = []

# Fit a quick calibration model across all regimes for this plot
calib_preds = {}
for q in quantile_levels:
    m = xgb.XGBRegressor(**BASE_PARAMS, quantile_alpha=q)
    m.fit(train_f[PRICE_FEATURES], train_f[TARGET], verbose=False)
    pred = m.predict(test_f[PRICE_FEATURES])
    empirical_coverage.append((test_f[TARGET].values <= pred).mean())

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], color=PC['text3'], lw=1, ls='--', label='Perfect calibration')
ax.plot(quantile_levels, empirical_coverage,
        color=PC['teal'], lw=2, marker='o', markersize=4,
        label='Observed coverage')
ax.fill_between(quantile_levels, quantile_levels, empirical_coverage,
                alpha=0.15, color=PC['teal'])
ax.set_xlabel('Quantile Level (nominal)')
ax.set_ylabel('Empirical Coverage (actual)')
ax.set_title('Quantile Calibration Plot — OMIE Spain', loc='left')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pc_calibration.png', dpi=150,
            bbox_inches='tight', facecolor=PC['navy'])
plt.show()
print('Closer to the diagonal = better calibrated intervals.')

---
## V. Forecast Visualisation

The forecast ribbon — the most useful chart for the trading desk.

In [ ]:
# ── 7-day forecast ribbon ─────────────────────────────────────
window = results['2024-06-01':'2024-06-07']

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle('OMIE Spain — Quantile Forecast Ribbon · June 2024',
             color=PC['text'], fontsize=13, y=0.99)

# Panel 1: Price forecast with P10/P90 envelope
ax = axes[0]
ax.fill_between(window.index, window['p10'], window['p90'],
                color=PC['teal'], alpha=0.15, label='P10–P90 interval')
ax.fill_between(window.index, window['p10'], window['p90'],
                where=window['in_band'] == 0,
                color=PC['crimson'], alpha=0.25, label='Actual outside band')
ax.plot(window.index, window['p50'],
        color=PC['teal'], lw=1.8, label='P50 (median forecast)')
ax.plot(window.index, window['p10'],
        color=PC['teal'], lw=0.7, ls=':', alpha=0.7)
ax.plot(window.index, window['p90'],
        color=PC['teal'], lw=0.7, ls=':', alpha=0.7)
ax.plot(window.index, window[TARGET],
        color=PC['text2'], lw=1.5, label='Actual price')
ax.set_title('Price Forecast (€/MWh) with Uncertainty Envelope', loc='left')
ax.set_ylabel('€/MWh')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Interval width — the uncertainty / position-sizing signal
ax2 = axes[1]
# Colour bars by regime
regime_c_map = {0: PC['gold'], 1: PC['teal'], 2: PC['crimson']}
bar_colors = [regime_c_map.get(r, PC['text3'])
              for r in window['pred_regime'].values]
ax2.bar(window.index, window['width'],
        color=bar_colors, alpha=0.75, width=pd.Timedelta('45min'))
ax2.axhline(window['width'].mean(), color=PC['text3'],
            lw=1, ls='--', label=f"Mean width {window['width'].mean():.1f} €/MWh")
ax2.set_title('Interval Width P90−P10 (€/MWh) — Position Sizing Signal',
              loc='left')
ax2.set_ylabel('Width (€/MWh)')

legend_els = [
    mpatches.Patch(color=PC['gold'],    label='Thermal-Marginal'),
    mpatches.Patch(color=PC['teal'],    label='Renewable-Dominant'),
    mpatches.Patch(color=PC['crimson'], label='Demand-Stress'),
    plt.Line2D([0],[0], color=PC['text3'], ls='--', label='Mean width'),
]
ax2.legend(handles=legend_els, fontsize=8, ncol=4)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pc_quantile_ribbon.png', dpi=150,
            bbox_inches='tight', facecolor=PC['navy'])
plt.show()

In [ ]:
# ── Single day D-1 output: 24-hour bar chart with error bars ──
day = results['2024-06-15':'2024-06-15']

fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('D-1 Forecast — 15 June 2024 · P10 / P50 / P90',
             color=PC['text'], fontsize=12)

hours = day.index.hour
rc    = [regime_c_map.get(r, PC['text3']) for r in day['pred_regime'].values]

# P50 bars
ax.bar(hours, day['p50'], color=rc, alpha=0.6, width=0.6, label='P50 forecast')

# Error bars spanning P10–P90
yerr_low  = day['p50'].values - day['p10'].values
yerr_high = day['p90'].values - day['p50'].values
ax.errorbar(hours, day['p50'],
            yerr=[yerr_low, yerr_high],
            fmt='none', color=PC['text2'],
            capsize=4, capthick=1.2, lw=1.2,
            label='P10–P90 interval')

# Actual
ax.plot(hours, day[TARGET], color=PC['text'],
        lw=2, marker='D', markersize=5, zorder=5,
        label='Actual')

legend_els = [
    mpatches.Patch(color=PC['gold'],    label='Thermal-Marginal'),
    mpatches.Patch(color=PC['teal'],    label='Renewable-Dominant'),
    mpatches.Patch(color=PC['crimson'], label='Demand-Stress'),
    plt.Line2D([0],[0], color=PC['text2'], lw=1.5, label='P10–P90'),
    plt.Line2D([0],[0], color=PC['text'], lw=2, marker='D', label='Actual'),
]
ax.legend(handles=legend_els, fontsize=8, ncol=5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('€/MWh')
ax.set_xticks(range(24))
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('pc_quantile_day.png', dpi=150,
            bbox_inches='tight', facecolor=PC['navy'])
plt.show()

---
## VI. Position Sizing Engine

This is where the quantile layer connects directly to trade sizing.

**Logic:**  
- Narrow interval + strong regime → full size  
- Wide interval + weak regime → reduce or pass  
- The multiplier is continuous, not binary

```
Sizing multiplier = Regime Conviction Score × Interval Confidence Score
```

In [ ]:
# ── Position sizing parameters ────────────────────────────────
SIZING = {
    'base_mw':              10,     # Base lot size (MW)
    'max_multiplier':       2.5,    # Max scale-up
    'width_floor':          5,      # Below this → maximum conviction (€/MWh)
    'width_cap':            60,     # Above this → minimum size (€/MWh)
    'min_regime_prob':      0.60,   # Minimum regime probability to trade
    'short_threshold':      -10,    # P50 below thermal floor by this → short signal
    'sunset_penetration':   0.30,   # Storage penetration threshold where edge decays
    'sunset_capture_ratio': 0.35,   # Surplus capture ratio that starts killing the edge
    'min_short_buffer':     0.15,   # Minimum residual edge to allow fresh shorts
}


def compute_position_size(row: pd.Series, params: dict) -> dict:
    """
    Computes position size for a single delivery hour.

    Returns dict with storage-aware conviction and position sizing.
    """
    pred_r = int(row['pred_regime'])
    prob_map = {0: row['prob_thermal'],
                1: row['prob_renewable'],
                2: row['prob_stress']}
    regime_conviction = prob_map.get(pred_r, 0.33)

    if regime_conviction < params['min_regime_prob']:
        return {'regime_conviction': regime_conviction,
                'interval_confidence': 0,
                'storage_buffer': 0,
                'combined_score': 0,
                'position_mw': 0,
                'direction': 'flat'}

    width = row['width']
    interval_confidence = 1 - np.clip(
        (width - params['width_floor'])
        / (params['width_cap'] - params['width_floor']),
        0, 1
    )

    storage_pressure = max(
        row.get('storage_penetration', 0) / params['sunset_penetration'],
        row.get('bess_capture_ratio', 0) / params['sunset_capture_ratio'],
    )
    storage_buffer = 1 - np.clip(storage_pressure, 0, 1)
    storage_adjustment = 0.35 + 0.65 * storage_buffer
    combined = regime_conviction * interval_confidence * storage_adjustment

    is_short_regime = pred_r == 1
    below_floor = row['p50'] < (
        row.get('thermal_floor', row['p50']) + params['short_threshold']
    )
    sunset_hit = (
        row.get('storage_sunset_flag', 0) == 1
        or storage_buffer < params['min_short_buffer']
    )

    if is_short_regime and below_floor and sunset_hit:
        direction = 'flat'
        position_mw = 0
    else:
        direction = 'short' if (is_short_regime and below_floor) else 'long'
        multiplier = 1 + (params['max_multiplier'] - 1) * combined
        position_mw = round(params['base_mw'] * multiplier, 1)

    return {
        'regime_conviction':   round(regime_conviction, 3),
        'interval_confidence': round(interval_confidence, 3),
        'storage_buffer':      round(storage_buffer, 3),
        'combined_score':      round(combined, 3),
        'position_mw':         position_mw,
        'direction':           direction,
    }


SUNSET_RESULT = estimate_storage_sunset(test_f, rsi_col='rsi')

# ── Apply to full test set ────────────────────────────────────
for col in [
    'thermal_floor', 'prob_thermal', 'prob_renewable', 'prob_stress',
    'storage_penetration', 'bess_capture_ratio', 'rsi_after_bess',
    'short_edge_score', 'storage_sunset_flag',
]:
    results[col] = test_f.loc[results.index, col]

sizing_output = results.apply(
    lambda row: compute_position_size(row, SIZING), axis=1
)
sizing_df = pd.DataFrame(list(sizing_output), index=results.index)
results   = pd.concat([results, sizing_df], axis=1)

print('Position sizing statistics:')
print(f'  Hours traded   : {(results["position_mw"] > 0).sum():,}')
print(f'  Hours flat     : {(results["position_mw"] == 0).sum():,}')
print(f'  Short signals  : {(results["direction"] == "short").sum():,}')
print(f'  Mean position  : {results["position_mw"].mean():.1f} MW')
print(f'  Max position   : {results["position_mw"].max():.1f} MW')
print()
print('Storage sunset estimate:')
print(f'  Penetration trigger : {SUNSET_RESULT.penetration_threshold:.2f}x p95 surplus')
print(f'  Capture trigger     : {SUNSET_RESULT.capture_ratio_threshold:.1%}')
print(f'  Online capacity     : {SUNSET_RESULT.online_capacity_threshold:.2f}')
print(f'  Edge retention      : {SUNSET_RESULT.edge_ratio:.1%} of baseline')
print(f'  Trigger source      : {SUNSET_RESULT.threshold_source}')
print()
results[['p10','p50','p90','width','regime_conviction',
         'interval_confidence','storage_buffer','combined_score',
         'position_mw','direction']].head(12).round(2)



In [ ]:
# ── Position sizing diagnostic chart ─────────────────────────
sample_w = results['2024-06-01':'2024-06-14']

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle('Position Sizing Engine — June 2024',
             color=PC['text'], fontsize=13, y=0.99)

# Panel 1: Price ribbon
axes[0].fill_between(sample_w.index, sample_w['p10'], sample_w['p90'],
                     color=PC['teal'], alpha=0.15)
axes[0].plot(sample_w.index, sample_w['p50'],
             color=PC['teal'], lw=1.5, label='P50')
axes[0].plot(sample_w.index, sample_w[TARGET],
             color=PC['text2'], lw=1, label='Actual')
axes[0].set_title('Price Forecast + Interval', loc='left')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Panel 2: Combined conviction score
axes[1].fill_between(sample_w.index, 0, sample_w['combined_score'],
                     color=PC['gold'], alpha=0.5)
axes[1].axhline(0.5, color=PC['text3'], lw=0.8, ls='--',
                label='Threshold 0.5')
axes[1].set_title('Combined Conviction Score (Regime × Interval)',
                  loc='left')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Panel 3: Position size in MW, coloured by direction
short_mask = sample_w['direction'] == 'short'
long_mask  = sample_w['direction'] == 'long'
axes[2].bar(sample_w.index[long_mask],
             sample_w['position_mw'][long_mask],
             color=PC['teal'],    alpha=0.75,
             width=pd.Timedelta('45min'), label='Long')
axes[2].bar(sample_w.index[short_mask],
            -sample_w['position_mw'][short_mask],
             color=PC['crimson'], alpha=0.75,
             width=pd.Timedelta('45min'), label='Short')
axes[2].axhline(0, color=PC['rule'], lw=0.8)
axes[2].set_title('Position Size (MW) — Short shown negative',
                  loc='left')
axes[2].set_ylabel('MW')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pc_position_sizing.png', dpi=150,
            bbox_inches='tight', facecolor=PC['navy'])
plt.show()

---
## VII. Updated Production Forecast Function

In [ ]:
def forecast_tomorrow_quantile(
    forecast_inputs: pd.DataFrame,
    regime_clf,
    scaler_reg,
    quantile_models: dict,
    regime_features: list,
    price_features:  list,
    sizing_params:   dict,
) -> pd.DataFrame:
    """
    Full D-1 production forecast.
    Input  : 24-row DataFrame of forecast features for tomorrow.
    Output : 24-row DataFrame with P10/P50/P90, uncertainty width,
             regime label, and position size recommendation.
    """
    inp = forecast_inputs.copy()

    # Stage 1: Regime
    X_reg        = scaler_reg.transform(inp[regime_features])
    inp['pred_regime']    = regime_clf.predict(X_reg)
    regime_prob           = regime_clf.predict_proba(X_reg)
    inp['prob_thermal']   = regime_prob[:, 0]
    inp['prob_renewable'] = regime_prob[:, 1]
    inp['prob_stress']    = regime_prob[:, 2]

    # Stage 2: Quantile forecasts
    for label in Q_LABELS:
        preds = np.full(len(inp), np.nan)
        for regime, regime_m in quantile_models.items():
            mask = inp['pred_regime'].values == regime
            if mask.sum() > 0 and label in regime_m:
                preds[mask] = regime_m[label].predict(
                    inp.loc[mask, price_features])
        inp[label] = preds

    inp['width'] = inp['p90'] - inp['p10']

    # Stage 3: Position sizing
    sizing = inp.apply(
        lambda row: compute_position_size(row, sizing_params), axis=1
    )
    sizing_df = pd.DataFrame(list(sizing), index=inp.index)

    out = pd.DataFrame({
        'hour':              inp.index.hour,
        'regime':            [regime_names[r] for r in inp['pred_regime']],
        'regime_prob':       inp[['prob_thermal','prob_renewable',
                                  'prob_stress']].max(axis=1).round(3),
        'p10':               inp['p10'].round(2),
        'p50':               inp['p50'].round(2),
        'p90':               inp['p90'].round(2),
        'width':             inp['width'].round(2),
        'conviction':        sizing_df['combined_score'],
        'direction':         sizing_df['direction'],
        'position_mw':       sizing_df['position_mw'],
    })

    return out


# ── Demo: run on sample day ───────────────────────────────────
sample_inputs = test_f['2024-07-15':'2024-07-15'][
    REGIME_FEATURES + PRICE_FEATURES + ['thermal_floor']
]

tomorrow = forecast_tomorrow_quantile(
    sample_inputs, regime_clf, scaler_reg,
    quantile_models, REGIME_FEATURES, PRICE_FEATURES, SIZING
)

print('D-1 Quantile Forecast — 15 July 2024')
print('─' * 78)
print(tomorrow.to_string(index=False))
print('─' * 78)
print(f'Hours with active position : {(tomorrow["position_mw"] > 0).sum()}')
print(f'Short signals              : {(tomorrow["direction"] == "short").sum()}')
print(f'Mean position size         : {tomorrow["position_mw"].mean():.1f} MW')
print(f'Mean interval width        : {tomorrow["width"].mean():.1f} €/MWh')

---
## VIII. Save & Export

In [ ]:
# ── Save quantile artifacts ───────────────────────────────────
quantile_artifacts = {
    **arts,   # everything from PROG01
    'price_features':        PRICE_FEATURES,
    'storage_features':      [f for f in STORAGE_FEATURES if f in PRICE_FEATURES],
    'quantile_models':       quantile_models,
    'sizing_params':         SIZING,
    'quantile_levels':       Q_LABELS,
    'storage_sunset_result': SUNSET_RESULT.__dict__,
}

with open('pc_spot_pricer_v2.pkl', 'wb') as f:
    pickle.dump(quantile_artifacts, f)

results.to_csv('pc_quantile_results.csv')

print('✓ Saved:')
print('  pc_spot_pricer_v2.pkl      — full model artifacts incl. quantile models')
print('  pc_quantile_results.csv    — test set P10/P50/P90 + position sizing')
print('  pc_calibration.png         — quantile calibration plot')
print('  pc_quantile_ribbon.png     — 7-day forecast ribbon')
print('  pc_quantile_day.png        — single day bar chart with error bars')
print('  pc_position_sizing.png     — conviction score + position size chart')
print()
print('─' * 55)
print('  PEACE CAPITAL — Quantile Layer Summary')
print('─' * 55)
cov = results['in_band'].mean()
print(f'  P10/P90 coverage  : {cov:.1%}  (target 80%)')
print(f'  Mean width        : {results["width"].mean():.1f} €/MWh')
print(f'  P50 MAE           : {results["p50_error"].abs().mean():.2f} €/MWh')
print(f'  Hours sized short : {(results["direction"]=="short").sum():,}')
print(f'  Sunset trigger    : {SUNSET_RESULT.penetration_threshold:.2f} penetration / {SUNSET_RESULT.capture_ratio_threshold:.1%} capture')
print('─' * 55)
print()
print('Next steps:')
print('  1. Replace synthetic fleet with hourly ESIOS BESS dispatch and online-capacity data')
print('  2. Feed node vulnerability scores into the congestion spread engine as an ICS input')
print('  3. Promote the sunset trigger into the strategy memo risk section')
print('  4. Add P5/P95 for tail-risk monitoring')
print('  5. Schedule forecast_tomorrow_quantile() as D-1 cron job')

